# DR 5 lớp @ 448 — huấn luyện trên Kaggle

`convnext_tiny` (tích chập) + `swin_t` (attention), rồi gộp. Ghép một CNN với một
transformer để ensemble không phải là hai bản sao của cùng một thiên kiến quy nạp.

## Cần upload gì

**Dataset 1 — dữ liệu:** upload `dr448_tables.rar` và **4** file `dr448_images_*.rar` trong `kaggle_upload/`.
Notebook tự giải nén nếu các file RAR còn nguyên trong `/kaggle/input` (cần khoảng **9 GB** đĩa tạm, không nạp cả archive vào RAM).

```
img448/
├── images/
│   ├── legacy/    24.877 ảnh
│   ├── mfiddr/    18.708 ảnh     <- MỚI, đừng quên Add Data
│   ├── drtid/      3.100 ảnh
│   └── deepdrid/   1.589 ảnh
├── manifest.csv
└── splits.csv
```

Tổng **48.274 ảnh / 28.015 nhóm**. Thiếu bất kỳ archive nào thì ô kiểm tra sẽ
dừng và nói rõ thiếu bao nhiêu ảnh của nguồn nào.

**Dataset 2 — code:** upload `dr448_code.rar` đã đóng gói lại sau khi sửa code.
Import bản notebook mới này vào Kaggle; code và dữ liệu đã giải nén sẵn cũng được hỗ trợ.

```
Source/training/{data,metrics,models,train,ensemble}.py
```

Cả hai gộp làm một dataset cũng chạy được — ô bên dưới tự dò tìm.

## Cài đặt notebook

| | |
|---|---|
| Accelerator | **GPU T4 x2** (code chỉ dùng 1 GPU; xem ghi chú cuối) |
| Internet | **On** (để `timm` tải trọng số pretrained) |
| Persistence | Variables and Files |

## Lưu ý về thời gian

Mỗi model mất ~3–4,5 giờ. Phiên GPU của Kaggle giới hạn 9 giờ chạy liên tục và
12 giờ/tuần trên một số tài khoản. **Nên chạy mỗi model một phiên**: dùng
*Save Version → Save & Run All (Commit)* để nó chạy nền, rồi tải checkpoint ra
làm dataset đầu vào cho phiên sau.


In [ ]:
# ── Môi trường ────────────────────────────────────────────────────────────────
import os, sys, shutil, subprocess, platform, collections
import torch

print(f"python {platform.python_version()} | torch {torch.__version__}")
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        p = torch.cuda.get_device_properties(i)
        print(f"  GPU {i}: {p.name}  {p.total_memory/2**30:.1f} GB  sm_{p.major}{p.minor}")
    # ĐỪNG tin torch.cuda.is_bf16_supported(): mặc định nó bật
    # including_emulation, và phép thử emulation chỉ cấp phát một tensor bf16 —
    # việc đó THÀNH CÔNG trên T4 (sm_75). Nhưng cuDNN không có engine conv bf16
    # dưới sm_80, nên autocast bf16 sẽ chết ở conv đầu tiên với
    #   RuntimeError: GET was unable to find an engine to execute this computation
    # Compute capability mới là phép thử thật.
    caps = [torch.cuda.get_device_capability(i) for i in range(torch.cuda.device_count())]
    native_bf16 = all(c[0] >= 8 for c in caps)
    print(f"  is_bf16_supported() nói: {torch.cuda.is_bf16_supported()}"
          f"   |  bf16 THẬT (sm_80+): {native_bf16}")
    print(f"  -> sẽ dùng {'bf16' if native_bf16 else 'fp16 + GradScaler'}")
else:
    raise SystemExit("Chưa bật GPU — Settings → Accelerator → GPU T4 x2")

print(f"CPU cores: {os.cpu_count()}")
free = shutil.disk_usage('/kaggle/working').free / 2**30
print(f"/kaggle/working còn trống: {free:.1f} GB")

for pkg in ("timm", "tqdm"):
    try:
        m = __import__(pkg)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)
        m = __import__(pkg)
        print(f"{pkg} {m.__version__} (vừa cài)")
    else:
        print(f"{pkg} {m.__version__}")

## Giải nén RAR nếu cần

Chạy một lần mỗi phiên. Dùng `bsdtar` (cài `libarchive-tools` nếu thiếu); các lần chạy lại bỏ qua archive đã hoàn thành. Không tải toàn bộ file nén vào RAM và không đưa ảnh vào output khi Save Version.


In [ ]:
# RAR được giải nén theo luồng xuống đĩa tạm, không vào /kaggle/working.
from pathlib import Path, PurePosixPath
import hashlib
import os
import shutil
import subprocess
import sys

INPUT = Path("/kaggle/input")
RAR_ROOT = Path("/tmp/dr448_rar")
RAR_ROOT.mkdir(parents=True, exist_ok=True)
EXTRACTED_ROOTS = []


def rar_extractor():
    for name in ("bsdtar", "tar"):
        tool = shutil.which(name)
        if tool:
            version = subprocess.run([tool, "--version"], capture_output=True, text=True)
            if "bsdtar" in version.stdout.lower() or "libarchive" in version.stdout.lower():
                return tool
    if sys.platform != "linux":
        raise RuntimeError("Cần bsdtar/libarchive-tools để giải nén RAR.")
    prefix = [] if os.geteuid() == 0 else ["sudo"]
    print("Cài libarchive-tools để đọc RAR (cần Internet)...", flush=True)
    subprocess.run(prefix + ["apt-get", "update", "-qq"], check=True)
    subprocess.run(prefix + ["apt-get", "install", "-y", "-qq",
                             "--no-install-recommends", "libarchive-tools"], check=True)
    tool = shutil.which("bsdtar")
    if not tool:
        raise RuntimeError("Không tìm thấy bsdtar sau khi cài libarchive-tools.")
    return tool


def extract_rar(archive, cache_root, tool):
    stat = archive.stat()
    signature = f"{archive.resolve()}:{stat.st_size}:{stat.st_mtime_ns}"
    version = hashlib.sha256(signature.encode()).hexdigest()[:16]
    destination = cache_root / f"{archive.stem}_{version}"
    destination.mkdir(parents=True, exist_ok=True)
    marker = destination / ".complete"
    if marker.is_file() and marker.read_text(encoding="utf-8") == signature:
        print(f"Đã giải nén: {archive.name}", flush=True)
        return destination

    listing = subprocess.run([tool, "-tf", str(archive)], check=True,
                             capture_output=True, text=True).stdout.splitlines()
    members = [PurePosixPath(name.replace("\\", "/")) for name in listing if name]
    if not members:
        raise RuntimeError(f"Archive rỗng: {archive}")
    for member in members:
        if (member.is_absolute() or ".." in member.parts
                or any(":" in part for part in member.parts)
                or member.parts[0] not in ("img448", "Source")):
            raise RuntimeError(f"Đường dẫn không hợp lệ trong {archive.name}: {member}")
    # Các RAR do package_for_kaggle.py tạo dùng -m0 (stored).
    required = stat.st_size + 512 * 2**20
    if shutil.disk_usage(destination).free < required:
        raise RuntimeError(f"Không đủ đĩa tạm để giải nén {archive.name}.")
    print(f"Giải nén {archive.name} ({stat.st_size / 2**30:.2f} GiB)...", flush=True)
    subprocess.run([tool, "-xf", str(archive), "-C", str(destination)], check=True)
    marker.write_text(signature, encoding="utf-8")
    return destination


archives = sorted(INPUT.rglob("dr448_*.rar"))
if archives:
    tool = rar_extractor()
    for archive in archives:
        EXTRACTED_ROOTS.append(extract_rar(archive, RAR_ROOT, tool))
    print(f"Đã sẵn sàng {len(EXTRACTED_ROOTS)} archive; dữ liệu nằm trong {RAR_ROOT}.")
else:
    print("Không có RAR thô; dùng dữ liệu/code đã giải nén trong /kaggle/input.")


## Dò tìm dữ liệu và code

Ô trước đã giải nén các RAR còn nguyên vào `/tmp`. Ô này tìm `manifest.csv` + `splits.csv`, `train.py` và các thư mục `images/<nguồn>` trong cả input lẫn các thư mục vừa giải nén.

Các nhánh được ghép bằng symlink tại `/tmp/img448_images`, không copy thêm ảnh vào output. Nếu gắn nhiều bản data/code cùng lúc, notebook dừng để tránh dùng nhầm; chỉ Add Input một phiên bản cho mỗi nguồn.


In [ ]:
# ── Dò tìm đường dẫn ──────────────────────────────────────────────────────────
from pathlib import Path

INPUT = Path("/kaggle/input")
SEARCH_ROOTS = [INPUT] + EXTRACTED_ROOTS

def src_label(path):
    # <archive>/img448/images/<source> -> "<archive>", để đối chiếu với Add Data
    return path.parents[2].name if len(path.parents) > 2 else str(path)


# Đường dẫn thật đã gặp: datasets/<user>/<dataset>/<archive>/Source/training/train.py
# là 6 tầng -- vừa đúng trần cũ. Để 9 cho có biên, chi phí chỉ là vài glob rỗng.
DEPTH = 9

def find(pattern, sibling=None, depth=DEPTH):
    # mọi đường dẫn khớp `pattern` trong vòng `depth` cấp, có `sibling` bên cạnh
    hits = set()
    for root in SEARCH_ROOTS:
        for d in range(depth + 1):
            hits.update(root.glob("/".join(["*"] * d + [pattern])))
    if sibling:
        hits = {p for p in hits if (p.parent / sibling).exists()}
    return sorted(hits)

manifests = find("manifest.csv", sibling="splits.csv")
trainers  = find("train.py", sibling="data.py")
if not manifests:
    raise SystemExit("Không thấy manifest.csv cạnh splits.csv trong /kaggle/input.\n"
                     f"Đang có: {[p.name for p in INPUT.iterdir()]}")
if not trainers:
    raise SystemExit("Không thấy train.py cạnh data.py trong /kaggle/input.\n"
                     f"Đang có: {[p.name for p in INPUT.iterdir()]}")

if len(manifests) != 1 or len(trainers) != 1:
    raise SystemExit(f"Có nhiều bản data/code. Gỡ dataset cũ khỏi Add Input rồi restart kernel.\n"
                     f"manifest: {manifests}\ntrain.py: {trainers}")

DATA_DIR = manifests[0].parent
CODE_DIR = trainers[0].parent
WORK     = Path("/kaggle/working/checkpoints"); WORK.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(CODE_DIR))

# Kaggle giải nén MỖI archive vào thư mục riêng, nên images/<nguồn> nằm rải rác ở
# nhiều nhánh và KHÔNG nằm cạnh manifest.csv. Tìm hết rồi gom bằng symlink.
found = {}
for root in SEARCH_ROOTS:
    for d in range(DEPTH + 1):
        for p in root.glob("/".join(["*"] * d + ["images"])):
            if not p.is_dir():
                continue
            for sub in p.iterdir():
                if sub.is_dir() and next(sub.glob("*.png"), None) is not None:
                    if sub.name in found and found[sub.name] != sub:
                        raise SystemExit(f"Nhiều thư mục ảnh cho {sub.name}: {found[sub.name]} và {sub}")
                    found[sub.name] = sub
if not found:
    raise SystemExit("Không thấy thư mục images/<nguồn>/ nào chứa .png.\n"
                     f"Đang có: {[p.name for p in INPUT.iterdir()]}")

IMAGE_ROOT = Path("/tmp/img448_images")
IMAGE_ROOT.mkdir(parents=True, exist_ok=True)
for name, src in sorted(found.items()):
    link = IMAGE_ROOT / name
    if link.is_symlink():
        if link.resolve() == src.resolve():
            continue
        link.unlink()  # chỉ thay symlink do notebook tạo, không xoá thư mục nguồn
    elif link.exists():
        raise SystemExit(f"{link} không phải symlink. Restart phiên trước khi đổi dataset.")
    try:
        link.symlink_to(src, target_is_directory=True)
    except OSError:  # cực hiếm; copy là phương án cuối, tốn 5,3 GB
        shutil.copytree(src, link)

print(f"data   {DATA_DIR}")
print(f"code   {CODE_DIR}")
print(f"out    {WORK}")
print(f"images {IMAGE_ROOT}  (gom từ {len(found)} nhánh)")
# Đếm ở thư mục gốc chứ không qua symlink: pathlib trước 3.13 không đi xuyên
# symlink khi glob bằng '**', nên rglob qua IMAGE_ROOT trả về 0 và trông như mất ảnh.
counts = {name: sum(1 for _ in src.glob("*.png")) for name, src in sorted(found.items())}
for name, c in counts.items():
    print(f"   {name:10} {c:>7,} ảnh   <- {src_label(found[name])}")
n = sum(counts.values())
print(f"\nảnh trên đĩa: {n:,}")

## Kiểm tra dữ liệu trước khi tiêu 4 giờ GPUBa câu hỏi, theo thứ tự mức độ nghiêm trọng nếu sai:1. Nhóm nào bắc qua hai split không? (rò rỉ — mọi con số sau đó vô nghĩa)2. File có đủ và đúng `448×448×3` không?3. Phân bố lớp có khớp với lúc dựng không?

In [ ]:
# ── Kiểm tra ──────────────────────────────────────────────────────────────────
import csv, collections, random
import numpy as np, cv2

man = list(csv.DictReader(open(DATA_DIR / "manifest.csv", encoding="utf-8-sig")))
spl = {r["image_id"]: r for r in csv.DictReader(open(DATA_DIR / "splits.csv", encoding="utf-8-sig"))}
print(f"manifest {len(man):,} | splits {len(spl):,}")
assert len(man) == len(spl), "manifest và splits không khớp số dòng"

# Chỉ chiều THIẾU mới là lỗi. Ảnh thừa trên đĩa vô hại -- chúng đơn giản là
# không được manifest tham chiếu, chẳng hạn khi dataset còn giữ bản dựng cũ.
want = collections.Counter(r["dataset_source"] for r in man)
lack = {k: want[k] - counts.get(k, 0) for k in want if want[k] > counts.get(k, 0)}
if lack:
    raise SystemExit(f"manifest cần {len(man):,} ảnh, đĩa có {n:,}.\n"
                     f"THIẾU theo nguồn: {lack}\n"
                     "-> chưa Add Data đủ các archive dr448_images_*")
if n > len(man):
    print(f"[chú ý] đĩa có {n - len(man):,} ảnh thừa ngoài manifest; bỏ qua chúng")

# 1. rò rỉ nhóm
gs = collections.defaultdict(set)
for r in man:
    gs[r["group_id"]].add(spl[r["image_id"]]["split"])
straddle = [g for g, s in gs.items() if len(s) > 1]
print(f"nhóm bắc qua split: {len(straddle)}  {'OK' if not straddle else '*** RÒ RỈ ***'}")
assert not straddle

# 2. hình học, trên mẫu ngẫu nhiên
from data import relocate
random.seed(0)
bad, fills = [], []
for r in random.sample(man, min(200, len(man))):
    p = relocate(r["processed_path"], IMAGE_ROOT)
    im = cv2.imdecode(np.fromfile(p, np.uint8), cv2.IMREAD_COLOR)
    if im is None or im.shape != (448, 448, 3):
        bad.append(p)
    else:
        fills.append((im.max(2) > 10).mean())
print(f"ảnh sai kích thước: {len(bad)}")
assert not bad
f = np.array(fills)
print(f"độ phủ khung: p05 {np.percentile(f,5):.3f}  trung vị {np.median(f):.3f}  p95 {np.percentile(f,95):.3f}"
      f"   (π/4 = {np.pi/4:.3f} nghĩa là đĩa nội tiếp chuẩn)")

# 3. phân bố
per = collections.defaultdict(collections.Counter)
for r in man:
    per[spl[r["image_id"]]["split"]][r["class_label"]] += 1
tot = collections.Counter(r["class_label"] for r in man)
print(f"\n{'split':7}{'ảnh':>8}   " + "  ".join(f"{c:>11}" for c in sorted(tot)))
for s in ("train", "val", "test"):
    t = sum(per[s].values())
    print(f"{s:7}{t:>8,}   " + "  ".join(f"{per[s][c]:>5,}/{100*per[s][c]/t:4.1f}%" for c in sorted(tot)))
print(f"{'tổng':7}{len(man):>8,}   " + "  ".join(f"{tot[c]:>5,}/{100*tot[c]/len(man):4.1f}%" for c in sorted(tot)))

In [ ]:
# ── Xem thử ảnh: một hàng cho mỗi mức độ ─────────────────────────────────────
import matplotlib.pyplot as plt

by_class = collections.defaultdict(list)
for r in man:
    by_class[r["class_label"]].append(r)

random.seed(1)
fig, axes = plt.subplots(5, 6, figsize=(15, 12.8))
for c in range(5):
    picks = random.sample(by_class[str(c)], min(6, len(by_class[str(c)])))
    for j, r in enumerate(picks):
        im = cv2.imdecode(np.fromfile(relocate(r["processed_path"], IMAGE_ROOT), np.uint8),
                          cv2.IMREAD_COLOR)
        axes[c, j].imshow(cv2.cvtColor(im, cv2.COLOR_BGR2RGB))
        axes[c, j].set_axis_off()
        if j == 0:
            axes[c, j].set_title(f"lớp {c}  (n={len(by_class[str(c)]):,})",
                                 loc="left", fontsize=11)
plt.tight_layout(); plt.show()

## Cấu hình chạy

`AMP` do chính notebook quyết, dựa trên compute capability đo ở ô đầu — **không**
phó thác cho `train.py`. T4 là sm_75 nên cuDNN không có kernel convolution bf16;
tin `torch.cuda.is_bf16_supported()` ở đó sẽ chết ngay conv đầu tiên với
`RuntimeError: GET was unable to find an engine to execute this computation`.
Quyết ở đây thì ô này chạy đúng kể cả khi dataset code vẫn là bản cũ.

`workers` đặt theo số core thật của máy Kaggle. FINDINGS ghi 14 worker ở 448 đã
gây `MemoryError`; trần đo được là 6, và máy Kaggle chỉ có 4 core nên 4 là đủ.

Batch: T4 có 16 GB nên rộng hơn nhiều so với RTX 3050 4,3 GB (nơi trần là
`swin@448 = 6`). Nếu gặp OOM thì hạ `BATCH` và tăng `ACCUM` tương ứng — tích của
hai số mới là thứ ảnh hưởng tới tối ưu hoá.

Cả hai model dùng `SAMPLER_STRENGTH=0.5`, `LOSS_WEIGHT_STRENGTH=0.0`, `PROGRESS=True`. Thanh `tqdm` hiện batch/total, tốc độ, ETA, loss và LR. `leave=False` ẩn thanh sau khi hoàn thành; log lưu chỉ còn dòng tổng kết epoch. Muốn xem hiệu ứng trực tiếp, chạy ô train trong phiên tương tác.


In [ ]:
# ── Cấu hình ──────────────────────────────────────────────────────────────────
from train import TrainConfig, train
import metrics as M

WORKERS = min(4, os.cpu_count() or 2)
BATCH, ACCUM = 12, 3          # effective batch = 36
SAMPLER_STRENGTH = 0.5
LOSS_WEIGHT_STRENGTH = 0.0   # không đổi loss trong lần thử sampler này
PROGRESS = True             # tqdm cập nhật batch trong từng epoch

# Notebook tự quyết precision từ compute capability đo ở ô đầu, thay vì phó thác
# cho train.py. Như vậy ô này chạy đúng dù dataset code là bản cũ hay mới -- và
# nó dùng phép thử đúng (sm_80+), không phải is_bf16_supported().
AMP = "bf16" if native_bf16 else "fp16"

RUNS = {
    # convnext hội tụ nhanh hơn và chịu được LR cao hơn swin
    "convnext_tiny": dict(epochs=20, learning_rate=2e-4),
    # transformer cần LR thấp hơn, và FINDINGS cho thấy swin cần nhiều epoch hơn
    "swin_t":        dict(epochs=30, learning_rate=1e-4),
}

def run(backbone, head="softmax", **overrides):
    cfg = TrainConfig(
        backbone=backbone, head=head, amp=AMP,
        batch_size=BATCH, accumulate=ACCUM, workers=WORKERS,
        **{"sampler_strength": SAMPLER_STRENGTH,
           "loss_weight_strength": LOSS_WEIGHT_STRENGTH,
           "progress": PROGRESS, **RUNS[backbone], **overrides},
    )
    print(f"\n{'='*78}\n{backbone} / {head}  "
          f"{cfg.epochs} epoch, batch {cfg.batch_size}x{cfg.accumulate}, "
          f"lr {cfg.learning_rate}, amp {cfg.amp}, sampler {cfg.sampler_strength}, "
          f"progress {cfg.progress}\n{'='*78}")
    return train(cfg, DATA_DIR / "manifest.csv", DATA_DIR / "splits.csv", WORK, IMAGE_ROOT)

print(f"workers {WORKERS} | batch {BATCH} x {ACCUM} = {BATCH*ACCUM} | amp {AMP}")
print(f"sampler_strength={SAMPLER_STRENGTH} | loss_weight_strength={LOSS_WEIGHT_STRENGTH} | progress={PROGRESS}")
for k, v in RUNS.items():
    print(f"  {k:14} {v}")

## Model 1 — convnext_tiny~20 epoch × **~11,3 phút ≈ 3,8 giờ** (tập train đã lên 33.819 ảnh, gấp 1,64 lần
lúc đo 413 s/epoch).

> Cân nhắc giảm xuống **12–15 epoch**. Ở lần chạy trước, convnext đứng yên từ
> epoch 14 và swin từ epoch 16 — comp dao động 0,0109 suốt 10 epoch cuối. Với dữ
> liệu nhiều hơn thì càng ít cần epoch, không phải nhiều hơn.Thanh tiến trình hiện `loss` trung bình đang chạy và `lr` hiện tại; nếu fp16 phảibỏ batch nào vì loss vô hạn thì có thêm `skip=<n>`. Bar tự xoá sau mỗi epoch, chỉđể lại dòng tóm tắt — 30 epoch không đọng lại 30 thanh chết.Trong log, `comp` là tiêu chí chọn checkpoint: `0,65·QWK + 0,35·balanced_accuracy`.Chọn theo QWK trần đã được đo là sai — epoch 9 (QWK 0,8516 / recall lớp 3 = 0,64)thua epoch 10 (QWK 0,8584 / recall lớp 3 = **0,47**) theo QWK, tức là vứt đúng thứmà balanced sampler được thêm vào để mua.

In [ ]:
summary_convnext = run("convnext_tiny")

## Model 2 — swin_t~30 epoch × **~14,2 phút ≈ 7,1 giờ** — sát trần 9 giờ của một phiên Kaggle, và
chưa tính thời gian TTA + hiệu chuẩn ở cuối. **Chạy riêng một phiên**, hoặc giảm
còn 15 epoch (~3,6 giờ) để convnext và swin vừa chung một phiên.448 là kích thước lớn duy nhất giữ cửa sổ 7 px của Swin V1 căn khớp:448/4 = 112 → các stage 112 / 56 / 28 / 14, đều chia hết cho 7. Ở 384 thì cả bốnstage đều phải đệm.> Nếu phiên trước đã hết giờ: bỏ qua ô convnext ở trên, chỉ chạy ô này.

In [ ]:
summary_swin = run("swin_t")

## Gộp hai modelTrung bình ở không gian **xác suất**, không phải logit — hai đầu có thể tự tin ởnhững mức khác nhau, nên trung bình logit thô sẽ ưu ái model nào tình cờ xuất sốlớn hơn.Trọng số trộn, temperature từng model và ngưỡng ordinal đều khớp **trên val** rồiđóng băng. Không thứ nào nhìn thấy test.

In [ ]:
# Cần cả hai file logits_*.npz đã tồn tại trong WORK.
# Nếu chạy hai phiên riêng, hãy Add Data phiên trước rồi copy sang WORK trước khi chạy ô này.
import subprocess

missing = [f"logits_{b}_softmax_val_tta.npz" for b in RUNS
           if not (WORK / f"logits_{b}_softmax_val_tta.npz").is_file()]
if missing:
    print("Thiếu:", missing)
    print("Chạy nốt model còn lại, hoặc copy .npz từ phiên trước vào", WORK)
else:
    subprocess.run([sys.executable, str(CODE_DIR / "ensemble.py"),
                    "--checkpoints", str(WORK),
                    "--runs", "convnext_tiny_softmax", "swin_t_softmax",
                    "--tta"], check=True)

## Bảng kết quảĐọc theo thứ tự: `argmax` là quyết định thô, `calibrated` là sau temperature +ngưỡng ordinal khớp trên val. `_tta` là trung bình qua 8 phép đối xứng của hìnhvuông — chúng không mất mát trên crop này (đã kiểm chứng 0 pixel mất trên 60 ảnh× 8 lượt) nên TTA thêm góc nhìn mà không bịa nội dung.

In [ ]:
# ── Tổng hợp ──────────────────────────────────────────────────────────────────
import json

rows = []
for f in sorted(WORK.glob("summary_*.json")):
    s = json.loads(f.read_text(encoding="utf-8"))
    name = f.stem.replace("summary_", "")
    for key, val in s["results"].items():
        for kind in ("argmax", "calibrated"):
            if kind in val:
                rows.append((name, key, kind, val[kind]))

ens = WORK / "ensemble_tta.json"
if ens.is_file():
    s = json.loads(ens.read_text(encoding="utf-8"))
    for key in ("test_argmax", "test_ordinal"):
        rows.append(("ENSEMBLE", key.replace("test_", "test/"), "-", s["results"][key]))
    print("trọng số trộn:", {k: round(v, 2) for k, v in s["weights"].items()})
    print()

hdr = f"{'run':24} {'split':10} {'kind':11} {'qwk':>7} {'bal':>7} {'f1':>7} {'comp':>7}   recall 0..4"
print(hdr); print("-" * len(hdr))
for name, key, kind, v in rows:
    rec = "/".join(f"{v[f'recall_{c}']:.2f}" for c in range(5))
    print(f"{name:24} {key:10} {kind:11} {v['qwk']:7.4f} {v['balanced_accuracy']:7.4f} "
          f"{v['macro_f1']:7.4f} {v['composite']:7.4f}   {rec}")

In [ ]:
# ── Đường cong huấn luyện ─────────────────────────────────────────────────────
import matplotlib.pyplot as plt

files = sorted(WORK.glob("summary_*.json"))
if files:
    fig, axes = plt.subplots(1, 3, figsize=(16, 4.2))
    for f in files:
        s = json.loads(f.read_text(encoding="utf-8"))
        h = s["history"]; e = [r["epoch"] for r in h]
        name = f.stem.replace("summary_", "")
        axes[0].plot(e, [r["train_loss"] for r in h], label=f"{name} train")
        axes[0].plot(e, [r["val_loss"] for r in h], "--", label=f"{name} val")
        axes[1].plot(e, [r["qwk"] for r in h], label=f"{name} qwk")
        axes[1].plot(e, [r["composite"] for r in h], "--", label=f"{name} comp")
        axes[2].plot(e, [r["recall_3"] for r in h], label=f"{name} lớp 3")
        axes[2].plot(e, [r["recall_1"] for r in h], "--", label=f"{name} lớp 1")
        axes[1].axvline(s["best_epoch"], color="grey", lw=0.8, ls=":")
    for ax, t in zip(axes, ("loss", "QWK vs composite", "recall lớp hiếm")):
        ax.set_title(t); ax.set_xlabel("epoch"); ax.legend(fontsize=8); ax.grid(alpha=.3)
    plt.tight_layout(); plt.show()

---

## Ghi chú

**val_loss tăng mà accuracy đứng yên là bình thường.** Cross-entropy đo *độ tự tin
đặt đúng chỗ* (`−log p`, không có chặn trên); QWK chỉ đo *lớp nào thắng*. Sau điểm
ngọt, model không đổi quyết định mà chỉ đẩy xác suất về 0/1 — những ca vốn đã sai
giờ sai một cách tự tin hơn. Đo được trên dự án này: convnext ep7 → ep19 val_loss
+22,2 % trong khi accuracy chỉ −2,2 % (chênh 10,2 lần). Đó là **overconfidence**,
và temperature scaling ở cuối chính là thuốc cho nó.

**Vì sao không dùng cả 2 GPU.** `DataParallel` nhân bản model mỗi bước và gom
output về GPU 0; ở 448 nó tăng RAM host mà không tăng thông lượng tương ứng. Nếu
muốn 2 GPU thì dùng `torchrun` với DDP — nhưng đừng đụng vào
`set_sharing_strategy('file_system')`: nó rò vào `/dev/shm`, thứ **không phải**
page cache thu hồi được, và kết thúc bằng OOM kill sau vài model.

**Muốn thử chuẩn hoá màu:** `run("convnext_tiny", colour_mode="ben_graham")`.
Ben Graham giảm biến thiên giữa ảnh −91 % (CLAHE chỉ −26 %) nhưng phá **màu tuyệt
đối**, mà xuất tiết cứng (vàng) và xuất huyết (đỏ) phân biệt bằng màu. Phải A/B,
không áp mù.

**Muốn thử đầu ordinal:** `run("convnext_tiny", head="coral")`. 4 logit tích luỹ
dùng chung một vector chiếu với 4 bias riêng — đó là thứ khiến P(grade > k) chỉ có
thể giảm khi k tăng.

**Sampler hiện tại:** `sampler_strength=0.5`, giữ `loss_weight_strength=0.0`.
Đây là một ablation riêng; chưa tự động thêm class-weighted loss.

**Rò rỉ còn lại.** Ở pHash ≤ 1 không cặp nào bắc qua split. Ở pHash ≤ 2 có 4.377
cặp bắc qua, nhưng chỉ **32,2 % cùng nhãn so với nền ngẫu nhiên 25,7 %** — vượt
nền 6,5 điểm, tức phần quy được cho trùng thật chỉ khoảng **285 cặp** trên 14.455
ảnh val+test. (Bộ trước: 49,9 % so với nền 28,6 %, vượt 21,3 điểm.) Tỉ lệ giảm vì
MFIDDR chụp 4 trường chuẩn hoá nên sinh nhiều cặp *trông giống mà khác người* —
xem FINDINGS.md §3b.

**Dấu vân tay nguồn — rủi ro lớn hơn rò rỉ.** Một bộ phân loại tuyến tính phân
biệt MFIDDR với phần còn lại **chỉ bằng đặc trưng ảnh, accuracy 0,99**. Mà tỉ lệ
MFIDDR khác nhau theo lớp (30 % ở lớp 0, **64 % ở lớp 1**, 19 % ở lớp 4), nên
"trông giống máy MFIDDR" tự nó là manh mối về nhãn. Test set rút từ cùng hỗn hợp
nên lối tắt này **làm đẹp số test mà bảng tổng không bắt được**. Phải đọc kết quả
kèm phân tích theo nguồn.
